# GitHub Repository Security Analysis

Batch-collects GitHub repositories and scans them for security vulnerabilities.

**Pipeline:**
1. Sample 100 repos per month (2017–2025) via GitHub Search API using `pushed:` date
2. Target personal (user-owned) repos with low star counts
3. Shallow-clone each repo (`--depth 1`) — `pushed:` ensures HEAD = that month's state
4. Run Semgrep for CWE/CVE detection + pip-audit/npm audit for dependency vulnerabilities
5. Save all results incrementally to CSV

**Scope:** Python, JavaScript, TypeScript, Ruby, PHP

All key parameters are adjustable in the CONFIG cell.

## 1. Setup and Configuration

In [ ]:
import sys, os

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    repo_url = "https://github.com/EthanCui2008/AAR-Research-Paper-AI-Effects.git"
    repo_dir = "/content/AAR-Research-Paper-AI-Effects"
    if not os.path.exists(repo_dir):
        os.system(f"git clone {repo_url} {repo_dir}")
    os.chdir(repo_dir)

    # Semgrep + npm (not in apt on Colab, install via pip/nvm)
    os.system(f"{sys.executable} -m pip install -q semgrep")
    os.system("npm --version >/dev/null 2>&1 || (curl -fsSL https://deb.nodesource.com/setup_20.x | bash - && apt-get install -y nodejs) >/dev/null 2>&1")

os.system(f"{sys.executable} -m pip install -q -r requirements.txt")
print("Environment ready.")

In [ ]:
import json
import csv
import calendar
import random
import subprocess
import shutil
import time
import threading
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional

import pandas as pd
from github import Github, RateLimitExceededException
from github.GithubException import GithubException
from tqdm.notebook import tqdm
from dotenv import load_dotenv

warnings.filterwarnings('ignore')
print("Imports loaded.")

In [ ]:
# ── Adjustable parameters ──────────────────────────────────────────
CONFIG = {
    # Sampling
    'repos_per_month': 100,              # repos to collect per monthly interval
    'max_stars': 50,                     # upper star limit (personal/low-star repos)
    'start_year': 2017,
    'end_year': 2025,

    # Languages currently analysed (interpreted)
    'languages': ['Python', 'JavaScript', 'TypeScript', 'Ruby', 'PHP'],

    # Scanning
    'clone_timeout': 300,                # git-clone timeout (seconds)
    'semgrep_timeout': 600,              # semgrep timeout per repo (seconds)
    'max_workers': 4,                    # parallel clone+scan threads

    # Paths
    'repos_dir': Path('./repos'),
    'results_dir': Path('./results'),
}

for d in [CONFIG['repos_dir'], CONFIG['results_dir']]:
    d.mkdir(exist_ok=True)

print("Config loaded.")

In [ ]:
# ── Quick test run ─────────────────────────────────────────────────
# Flip TEST_MODE to True and run all cells — uses a tiny sample and
# separate output dir so it won't touch the real results.
TEST_MODE = True

if TEST_MODE:
    CONFIG['repos_per_month'] = 5
    CONFIG['results_dir'] = Path('./results_test')
    CONFIG['results_dir'].mkdir(exist_ok=True)
    print(f"TEST MODE: repos_per_month={CONFIG['repos_per_month']}, output -> {CONFIG['results_dir']}")

In [ ]:
GITHUB_TOKEN = None

if IN_COLAB:
    try:
        from google.colab import userdata
        GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
    except Exception:
        pass

if not GITHUB_TOKEN:
    load_dotenv('keys.env')
    GITHUB_TOKEN = os.getenv('GITHUB_TOKEN')

if not GITHUB_TOKEN:
    raise ValueError(
        "GitHub token not found!\n"
        "  Local : create keys.env with GITHUB_TOKEN=ghp_...\n"
        "  Colab : add GITHUB_TOKEN to Secrets (key icon)."
    )

github_client = Github(GITHUB_TOKEN, per_page=100)

def _get_core_rate(client):
    """Extract core rate-limit object across PyGithub versions."""
    rl = client.get_rate_limit()
    # v2.x: rl.core  |  v1.x: rl.rate  |  fallback: rl itself
    for attr in ('core', 'rate'):
        obj = getattr(rl, attr, None)
        if obj is not None and hasattr(obj, 'remaining'):
            return obj
    return rl

rate = _get_core_rate(github_client)
print(f"GitHub API — {rate.remaining}/{rate.limit} requests left, resets {rate.reset}")

## 2. Repository Collection

Repos are fetched per monthly interval across all configured languages using
the `pushed:` date qualifier (captures repos whose last activity was in that
month). Only user-owned repos with low star counts are kept. Results are saved
to `results/repositories.csv`.

In [ ]:
def _time_intervals(start_year: int, end_year: int) -> list:
    """Generate (start_date, end_date) for each month from start_year-01 to end_year-12."""
    intervals = []
    for year in range(start_year, end_year + 1):
        for month in range(1, 13):
            s = f"{year}-{month:02d}-01"
            last_day = calendar.monthrange(year, month)[1]
            e = f"{year}-{month:02d}-{last_day:02d}"
            intervals.append((s, e))
    return intervals


class RepositoryCollector:
    """Collects GitHub repos: monthly intervals, pushed: filter, user-owned, low stars."""

    def __init__(self, client: Github, config: dict):
        self.client = client
        self.cfg = config
        self.csv_path = config['results_dir'] / 'repositories.csv'

    def _wait_for_rate_limit(self):
        rate = _get_core_rate(self.client)
        wait = (rate.reset - datetime.utcnow()).total_seconds() + 10
        if wait > 0:
            print(f"  Rate-limited -- sleeping {wait:.0f}s ...")
            time.sleep(wait)

    def _search_interval(self, lang: str, start: str, end: str) -> list:
        """Search for user-owned repos pushed in [start, end] for one language."""
        query = f"language:{lang} stars:0..{self.cfg['max_stars']} pushed:{start}..{end}"
        repos = []
        try:
            results = self.client.search_repositories(query=query, sort='stars', order='asc')
            for repo in results:
                if repo.owner.type != 'User':
                    continue
                repos.append({
                    'full_name': repo.full_name,
                    'name': repo.name,
                    'owner': repo.owner.login,
                    'owner_type': repo.owner.type,
                    'url': repo.html_url,
                    'clone_url': repo.clone_url,
                    'language': repo.language,
                    'stars': repo.stargazers_count,
                    'forks': repo.forks_count,
                    'created_at': repo.created_at.isoformat(),
                    'pushed_at': repo.pushed_at.isoformat(),
                    'description': (repo.description or '')[:200],
                    'interval_start': start,
                    'interval_end': end,
                })
                if len(repos) >= 200:
                    break
        except RateLimitExceededException:
            self._wait_for_rate_limit()
            return self._search_interval(lang, start, end)
        except GithubException as e:
            print(f"  API error ({lang} {start}): {e.status}")
        return repos

    def collect(self) -> pd.DataFrame:
        """Fetch repos per monthly interval, sample repos_per_month from each."""
        intervals = _time_intervals(self.cfg['start_year'], self.cfg['end_year'])
        languages = self.cfg['languages']
        n_per_month = self.cfg['repos_per_month']
        rng = random.Random(42)

        sampled = []
        total = len(intervals) * len(languages)
        with tqdm(total=total, desc="Fetching repos") as pbar:
            for start, end in intervals:
                pool = []
                for lang in languages:
                    pbar.set_postfix_str(f"{lang} {start}")
                    batch = self._search_interval(lang, start, end)
                    pool.extend(batch)
                    pbar.update(1)

                # Deduplicate within this month
                seen = set()
                unique = []
                for r in pool:
                    if r['full_name'] not in seen:
                        seen.add(r['full_name'])
                        unique.append(r)

                # Sample
                if len(unique) <= n_per_month:
                    sampled.extend(unique)
                else:
                    sampled.extend(rng.sample(unique, n_per_month))

        df = pd.DataFrame(sampled).drop_duplicates(subset='full_name').reset_index(drop=True)
        df.to_csv(self.csv_path, index=False)

        counts = df.groupby('interval_start').size()
        print(f"Saved {len(df)} repos -> {self.csv_path}")
        print(f"  Months: {len(counts)},  per-month: "
              f"min={counts.min()} / median={int(counts.median())} / max={counts.max()}")
        return df

In [ ]:
collector = RepositoryCollector(github_client, CONFIG)
repos_df = collector.collect()

## 3. Security & Dependency Scanning

Each repo is shallow-cloned once, then scanned with:
1. **Semgrep** (`--config auto`) for CWE/CVE detection
2. **pip-audit** / **npm audit** for dependency vulnerabilities

Results are saved incrementally to `semgrep_findings.csv` and
`deprecated_dependencies.csv`. Errors go to `scan_errors.csv`.

In [ ]:
class RepoScanner:
    """Clone each repo once, run Semgrep + dependency scans, save incrementally."""

    FINDING_FIELDS = [
        'repo_name', 'language', 'stars', 'interval_start', 'interval_end',
        'rule_id', 'severity', 'message', 'file', 'line_start', 'line_end',
        'cwe', 'owasp', 'cve', 'confidence', 'category',
    ]
    DEP_FIELDS = [
        'repo_name', 'language', 'stars', 'interval_start', 'interval_end',
        'package', 'version', 'vulnerability_id', 'severity',
        'description', 'ecosystem', 'fix_versions',
    ]

    def __init__(self, config: dict):
        self.cfg = config
        self.findings_csv = config['results_dir'] / 'semgrep_findings.csv'
        self.deps_csv = config['results_dir'] / 'deprecated_dependencies.csv'
        self.errors_csv = config['results_dir'] / 'scan_errors.csv'
        self._sg_lock = threading.Lock()
        self._dep_lock = threading.Lock()
        self._err_lock = threading.Lock()
        self._env = {**os.environ, 'PYTHONUTF8': '1', 'PYTHONIOENCODING': 'utf-8'}
        self._reset_csvs()

    def _reset_csvs(self):
        with open(self.findings_csv, 'w', newline='', encoding='utf-8') as f:
            csv.writer(f).writerow(self.FINDING_FIELDS)
        with open(self.deps_csv, 'w', newline='', encoding='utf-8') as f:
            csv.writer(f).writerow(self.DEP_FIELDS)
        with open(self.errors_csv, 'w', newline='', encoding='utf-8') as f:
            csv.writer(f).writerow(['repo_name', 'stage', 'error'])

    def _log_error(self, repo_name: str, stage: str, error: str):
        with self._err_lock:
            with open(self.errors_csv, 'a', newline='', encoding='utf-8') as f:
                csv.writer(f).writerow([repo_name, stage, error[:300]])

    def _warm_rules_cache(self):
        """Download semgrep rules once before parallel workers start."""
        print("Warming semgrep rules cache ...", flush=True)
        tmp = self.cfg['repos_dir'] / '_semgrep_warmup'
        tmp.mkdir(parents=True, exist_ok=True)
        dummy = tmp / 'test.py'
        dummy.write_text('x = 1\n', encoding='utf-8')
        try:
            r = subprocess.run(
                ['semgrep', '--config', 'auto', '--json',
                 tmp.resolve().as_posix()],
                capture_output=True, timeout=120, env=self._env,
            )
            if r.returncode == 0:
                print("  Rules cached.", flush=True)
            else:
                stderr = r.stderr.decode('utf-8', errors='replace')[:500]
                print(f"  Cache warning (rc={r.returncode}): {stderr}", flush=True)
        except Exception as e:
            print(f"  Cache warm-up failed: {e}", flush=True)
        finally:
            shutil.rmtree(tmp, ignore_errors=True)

    # -- clone -----------------------------------------------------------
    def _clone(self, repo: dict) -> Optional[Path]:
        dest = self.cfg['repos_dir'] / repo['full_name'].replace('/', '_')
        if dest.exists():
            return dest
        dest.parent.mkdir(parents=True, exist_ok=True)
        try:
            r = subprocess.run(
                ['git', 'clone', '--depth', '1', repo['clone_url'], str(dest)],
                capture_output=True, timeout=self.cfg['clone_timeout'],
            )
            if r.returncode != 0:
                stderr = r.stderr.decode('utf-8', errors='replace')[:200]
                self._log_error(repo['full_name'], 'clone', f"rc={r.returncode}: {stderr}")
                if dest.exists():
                    shutil.rmtree(dest, ignore_errors=True)
                return None
            return dest
        except subprocess.TimeoutExpired:
            if dest.exists():
                shutil.rmtree(dest, ignore_errors=True)
            return None
        except Exception as e:
            self._log_error(repo['full_name'], 'clone', str(e))
            if dest.exists():
                shutil.rmtree(dest, ignore_errors=True)
            return None

    # -- semgrep scan ----------------------------------------------------
    def _scan_semgrep(self, repo_path: Path, repo_name: str) -> list[dict]:
        posix_path = repo_path.resolve().as_posix()
        try:
            r = subprocess.run(
                ['semgrep', '--config', 'auto', '--json',
                 '--timeout', '60', '--max-target-bytes', '1000000',
                 posix_path],
                capture_output=True,
                timeout=self.cfg['semgrep_timeout'],
                cwd=posix_path,
                env=self._env,
            )
            stdout = r.stdout.decode('utf-8', errors='replace')
            stderr = r.stderr.decode('utf-8', errors='replace')
        except subprocess.TimeoutExpired:
            self._log_error(repo_name, 'semgrep', 'timed out')
            return []
        except Exception as e:
            self._log_error(repo_name, 'semgrep', str(e))
            return []

        if not stdout:
            self._log_error(repo_name, 'semgrep',
                            f"empty stdout, rc={r.returncode}, stderr={stderr[:300]}")
            return []
        try:
            data = json.loads(stdout)
        except json.JSONDecodeError as e:
            self._log_error(repo_name, 'semgrep',
                            f"bad JSON: {e}, stdout[:200]={stdout[:200]}")
            return []

        for err in data.get('errors', []):
            msg = err.get('message', '') or err.get('long_msg', '') or str(err)
            self._log_error(repo_name, 'semgrep-error', msg[:300])

        findings = []
        for hit in data.get('results', []):
            meta = hit.get('extra', {}).get('metadata', {})
            cwe_raw = meta.get('cwe', [])
            cve_raw = meta.get('cve', '') or meta.get('references', '')
            findings.append({
                'rule_id': hit.get('check_id', ''),
                'severity': hit.get('extra', {}).get('severity', ''),
                'message': (hit.get('extra', {}).get('message', '') or '')[:300],
                'file': hit.get('path', ''),
                'line_start': hit.get('start', {}).get('line', 0),
                'line_end': hit.get('end', {}).get('line', 0),
                'cwe': ','.join(cwe_raw) if isinstance(cwe_raw, list) else str(cwe_raw),
                'owasp': ','.join(meta.get('owasp', [])) if isinstance(meta.get('owasp'), list) else str(meta.get('owasp', '')),
                'cve': cve_raw if isinstance(cve_raw, str) else ','.join(cve_raw),
                'confidence': meta.get('confidence', ''),
                'category': meta.get('category', ''),
            })
        return findings

    def _append_semgrep(self, repo: dict, findings: list[dict]):
        with self._sg_lock:
            with open(self.findings_csv, 'a', newline='', encoding='utf-8') as f:
                w = csv.DictWriter(f, fieldnames=self.FINDING_FIELDS)
                for fd in findings:
                    fd.update({
                        'repo_name': repo['full_name'],
                        'language': repo.get('language', ''),
                        'stars': repo.get('stars', ''),
                        'interval_start': repo.get('interval_start', ''),
                        'interval_end': repo.get('interval_end', ''),
                    })
                    w.writerow(fd)

    # -- dependency scan -------------------------------------------------
    def _scan_python(self, req_file: Path, repo_name: str) -> list[dict]:
        vulns = []
        try:
            r = subprocess.run(
                ['pip-audit', '-r', str(req_file), '--format', 'json'],
                capture_output=True, timeout=120, env=self._env,
            )
            stdout = r.stdout.decode('utf-8', errors='replace')
            if not stdout:
                return []
            for dep in json.loads(stdout).get('dependencies', []):
                for v in dep.get('vulns', []):
                    vulns.append({
                        'package': dep.get('name', ''),
                        'version': dep.get('version', ''),
                        'vulnerability_id': v.get('id', ''),
                        'severity': v.get('fix_versions', [''])[0] if v.get('fix_versions') else '',
                        'description': (v.get('description', '') or '')[:300],
                        'ecosystem': 'python',
                        'fix_versions': ','.join(v.get('fix_versions', [])),
                    })
        except json.JSONDecodeError as e:
            self._log_error(repo_name, 'pip-audit', f"bad JSON: {e}")
        except Exception as e:
            self._log_error(repo_name, 'pip-audit', str(e))
        return vulns

    def _scan_npm(self, pkg_dir: Path, repo_name: str) -> list[dict]:
        vulns = []
        try:
            r = subprocess.run(
                ['npm', 'audit', '--json'],
                capture_output=True, timeout=120,
                cwd=str(pkg_dir), env=self._env,
            )
            stdout = r.stdout.decode('utf-8', errors='replace')
            if not stdout:
                return []
            for vid, v in json.loads(stdout).get('vulnerabilities', {}).items():
                vulns.append({
                    'package': v.get('name', vid),
                    'version': v.get('range', ''),
                    'vulnerability_id': vid,
                    'severity': v.get('severity', ''),
                    'description': (v.get('title', '') or '')[:300],
                    'ecosystem': 'npm',
                    'fix_versions': '',
                })
        except json.JSONDecodeError as e:
            self._log_error(repo_name, 'npm-audit', f"bad JSON: {e}")
        except Exception as e:
            self._log_error(repo_name, 'npm-audit', str(e))
        return vulns

    def _scan_dependencies(self, repo_path: Path, repo_name: str) -> list[dict]:
        vulns = []
        for p in repo_path.rglob('requirements.txt'):
            vulns.extend(self._scan_python(p, repo_name))
        for p in repo_path.rglob('package.json'):
            if 'node_modules' not in str(p):
                vulns.extend(self._scan_npm(p.parent, repo_name))
        return vulns

    def _append_deps(self, repo: dict, vulns: list[dict]):
        with self._dep_lock:
            with open(self.deps_csv, 'a', newline='', encoding='utf-8') as f:
                w = csv.DictWriter(f, fieldnames=self.DEP_FIELDS)
                for v in vulns:
                    v.update({
                        'repo_name': repo['full_name'],
                        'language': repo.get('language', ''),
                        'stars': repo.get('stars', ''),
                        'interval_start': repo.get('interval_start', ''),
                        'interval_end': repo.get('interval_end', ''),
                    })
                    w.writerow(v)

    # -- single-repo worker ----------------------------------------------
    def _process_one(self, repo: dict) -> tuple:
        """Clone, scan semgrep + deps, return (n_findings, n_vulns)."""
        n_findings = n_vulns = 0
        try:
            path = self._clone(repo)
            if path is None:
                return (0, 0)
            findings = self._scan_semgrep(path, repo['full_name'])
            if findings:
                self._append_semgrep(repo, findings)
            n_findings = len(findings)
            vulns = self._scan_dependencies(path, repo['full_name'])
            if vulns:
                self._append_deps(repo, vulns)
            n_vulns = len(vulns)
        except Exception as exc:
            self._log_error(repo['full_name'], 'scan', str(exc))
        finally:
            dest = self.cfg['repos_dir'] / repo['full_name'].replace('/', '_')
            shutil.rmtree(dest, ignore_errors=True)
        return (n_findings, n_vulns)

    # -- public API ------------------------------------------------------
    def scan_all(self, repos_df: pd.DataFrame):
        self._warm_rules_cache()
        print(f"Repos to scan: {len(repos_df)}")

        workers = self.cfg.get('max_workers', 4)
        total_findings = total_vulns = 0

        with ThreadPoolExecutor(max_workers=workers) as pool:
            futures = {}
            for _, row in repos_df.iterrows():
                f = pool.submit(self._process_one, row.to_dict())
                futures[f] = row['full_name']

            with tqdm(total=len(futures), desc=f"Scanning ({workers} workers)") as pbar:
                for f in as_completed(futures):
                    name = futures[f]
                    try:
                        nf, nv = f.result()
                        total_findings += nf
                        total_vulns += nv
                    except Exception as exc:
                        self._log_error(name, 'future', str(exc))
                    pbar.set_postfix(findings=total_findings, vulns=total_vulns)
                    pbar.update(1)

        errs_df = pd.read_csv(self.errors_csv)
        print(f"\nDone: {total_findings} findings, {total_vulns} dep vulns across {len(repos_df)} repos")
        print(f"  Errors: {len(errs_df)}  (see {self.errors_csv})")
        print(f"  {self.findings_csv}")
        print(f"  {self.deps_csv}")

In [ ]:
scanner = RepoScanner(CONFIG)
scanner.scan_all(repos_df)

## 4. Output Summary

In [ ]:
print("Output CSVs:")
for p in sorted(CONFIG['results_dir'].glob('*.csv')):
    size = p.stat().st_size
    print(f"  {p.name:40s}  {size:>10,} bytes")